In [1]:
import rosbag
import xml.etree.ElementTree as ET
from pathlib import Path
from pprint import pprint
from cv_bridge import CvBridge
import cv2,re
from sensor_msgs.msg import Image
import numpy as np
import matplotlib.pylab as plt
import open3d as o3d
from tqdm import tqdm
import math,copy
import os
from numba import jit
import scipy
from scipy.spatial.transform import Rotation
import pywavefront
import sensor_msgs.point_cloud2 as pc2
import ctypes,time
import struct
import Ros_Analysis
from collections import defaultdict
import copy,json

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/usr/local/lib/python3.8/dist-packages/numpy/core/getlimits.py:518: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.8/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.8/dist-packages/numpy/core/getlimits.py:518: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.8/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


# joint中有限制axes，能用那个乘上旋转矩阵变换的欧拉角这样就能用上运动学了
# 逐渐层debug

In [2]:
#所有的数据都集中在这里，或许用配置文件会更好？
#path 还有os.path中的数据，如果有/开头的路径，会直接去掉前面的所有路径
class path_data_class:
    def __init__(self):
        self.bag_name = Path("banana")
        self.folder_path = Path("/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF")
        self.urdf_path = self.folder_path / Path("bimanual_srhand_ur.urdf")
        self.bag_path = self.folder_path / Path(self.bag_name) / Path(str(self.bag_name) + ".bag")
        self.output_folder = self.folder_path/ Path(self.bag_name)  / Path("output")
        self.ros_path_prefix = Path("/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF")#如果搬运到另一台电脑上，要修改这个文件名
        self.rgb_timestamp_path = self.folder_path / Path(self.bag_name) 
        self.merge_mesh_output_folder = self.folder_path/ Path(self.bag_name)  / Path("merge_mesh/")
        self.gobal_position_folder = self.folder_path / Path(self.bag_name)  / Path("global_name_position/")
        self.pinhole_camera_parameters_path = Path("/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF/camera_params.json")
        self.TF_path = self.folder_path / Path(self.bag_name)  / Path("TF/")
        self.baginfo_path = self.folder_path / Path(self.bag_name) / Path("bag_info.txt")


class param_collect_class:
    def __init__(self):
        pass
path_data = path_data_class()

param_collect = param_collect_class()


setattr(param_collect,"bag_data",rosbag.Bag(path_data.bag_path))
print(path_data.gobal_position_folder)



/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF/banana/global_name_position


In [3]:
def baginfo_write(bag_data,path):
    bag_info = bag_data.get_type_and_topic_info()
    with open(path,"w+") as info_write:
        for item in bag_info:
            info_write.write(str(item))


baginfo_write(param_collect.bag_data, path_data.baginfo_path)

In [4]:
#save rgb camera timestamp
# you have to make the timestamp in this way
#or the timestamp wounld be very big
#你接下来的处理可能会经常出现下溢出
def get_rgbimage_timestamp(bag_data,path):
    if not os.path.exists(path):
        os.makedirs(path)
    path = path / "rgbimage_timestamp.txt"
    if os.path.exists(path): 
        return np.loadtxt(path,dtype=np.int64)
    else:
        time_topics = ['/cam0/rgb/image_raw']
        rgb_timestamp = []

        for _,msg,t in bag_data.read_messages(topics=time_topics):
            t = np.float128(str(t))
            rgb_timestamp.append(t)
        rgb_timestamp = np.array(rgb_timestamp)
        np.savetxt(path,rgb_timestamp,fmt="%f")
        return rgb_timestamp


setattr(param_collect,"rgb_timestamp",
        get_rgbimage_timestamp(param_collect.bag_data,path_data.rgb_timestamp_path))



/tmp/ipykernel_35976/3556955800.py:10: DeprecationWarning: loadtxt(): Parsing an integer via a float is deprecated.  To avoid this warning, you can:
    * make sure the original data is stored as integers.
    * use the `converters=` keyword argument.  If you only use
      NumPy 1.23 or later, `converters=float` will normally work.
    * Use `np.loadtxt(...).astype(np.int64)` parsing the file as
      floating point and then convert it.  (On all NumPy versions.)
  (Deprecated NumPy 1.23)
  return np.loadtxt(path,dtype=np.int64)


In [5]:
def build_TFtree(bag_data)->dict:
    global_name_postion =  {}
    TF_tree = defaultdict(set)
    link_names = set()
    num = 0
    tf_topics = ['/tf', '/tf_static']#有些TF，在TF中是存在的，但是
    for _,msg,_ in bag_data.read_messages(topics=tf_topics):
        for transform in msg.transforms:
            frame_id = transform.header.frame_id.replace('/','')
            chind_frame_id = transform.child_frame_id.replace('/','')
            link_names.add(frame_id)
            link_names.add(chind_frame_id)
            TF_tree[frame_id].add(chind_frame_id)
        num += 1
        if num >= 2000:#500个msg以上就能构建tftree 这里的值设的不够大，有可能会影响后面
            break
    global_name_postion = {name:None for name in link_names}
    print(TF_tree)
    return TF_tree,global_name_postion

setattr(param_collect,"TF_tree",None)
setattr(param_collect,"global_name_postion",None)

param_collect.TF_tree,param_collect.global_name_postion = build_TFtree(param_collect.bag_data)

#为什么我的gloabal会是这么大的，跟我自己手算的是差了10倍

defaultdict(<class 'set'>, {'ra_base': {'ra_tool0_controller'}, 'right_user_wrist': {'right_user_forearm'}, 'tracker_0': {'right_user_palm'}, 'rh_palm': {'rh_ffknuckle_fixed', 'rh_mfknuckle', 'rh_lf_palm_fixed', 'rh_mfknuckle_fixed', 'rh_ffknuckle', 'rh_lfmetacarpal', 'rh_imu', 'rh_thbase', 'rh_mf_palm_fixed', 'rh_lfmetacarpal_fixed', 'rh_rf_palm_fixed', 'rh_thbase_fixed', 'rh_manipulator', 'rh_mf_knuckle_glove', 'rh_rfknuckle'}, 'world': {'ra_base_link', 'sr_vive_root'}, 'polhemus_base_0': {'polhemus_station_0', 'rh_user_forearm', 'polhemus_station_2', 'polhemus_station_5', 'polhemus_station_4', 'polhemus_station_3', 'polhemus_station_1'}, 'cam3_camera_base': {'cam3_camera_body', 'cam3_depth_camera_link', 'cam3_camera_visor'}, 'cam2_camera_base': {'cam2_camera_visor', 'cam2_camera_body', 'cam2_depth_camera_link'}, 'cam1_camera_base': {'cam1_camera_body', 'cam1_depth_camera_link', 'cam1_camera_visor'}, 'cam0_camera_base': {'cam2_camera_base', 'cam0_camera_visor', 'cam3_camera_base', 'c

In [6]:
#如果后面tree的挂载出了问题，那么就把整个tree给画出来
def mount_data2TFtree(TF_tree:dict,TF_path):
    for key in TF_tree.keys():
        temp_set = TF_tree[key]
        TF_tree[key] = {item:[] for item in temp_set}
    folder = TF_path
    for filename in os.listdir(folder):
        match = re.match(r"(.+) -> (.+)\.txt",filename)
        link1 = match.group(1)
        link2 = match.group(2)
        #print(link1," ",link2)
        #print(type(TF_tree[link1]))
        transform = np.loadtxt(folder / Path(filename),dtype= np.float128)
        if transform.shape[0] == 8:
            transform = transform.reshape(1,8)
        TF_tree[link1][link2] = transform
    #专门添加的机械臂到相机之间的transoform
    #这里添加的这个，是会永久改变，后面不刷新的
    TF_tree['ra_base_link']['cam0_camera_base'] = np.array([[999, 1.97166146, -0.14104375 ,0.51574793, -0.30350154 , 0.12796092,  0.93549973 , 0.12787913]])
    #如果后面tree的挂载出了问题，那么就把整个tree给画出来

    return TF_tree

param_collect.TF_tree = mount_data2TFtree(param_collect.TF_tree,path_data.TF_path)



In [7]:
# import pydot
# def viz_TFtree(TF_tree,output_path):
#     graph = pydot.Dot(graph_type='digraph')
#     for key, value in TF_tree.items():
#         for child in value.keys():
#             graph.add_edge(pydot.Edge(key, child))
#         graph.write_pdf(output_path / Path("TFTREEviz.pdf"))

# viz_TFtree(param_collect.TF_tree,path_data.folder_path)

In [8]:
#init all the mesh in the urdf according to the right position in urdf
#urdf找到在TF tree中有节点的mesh
#以后obj的全部操作都由open3d进行
#i found that, not erery function support the path varivable

def euler_translation2transform(mesh_rpy,mesh_xyz):#这里面的xyz是以米为单位的吗
    mesh_transform = np.identity(4)
    mesh_rotation = Rotation.from_euler('xyz', mesh_rpy, degrees=False).as_matrix()
    mesh_transform[:3, :3] = mesh_rotation
    mesh_transform[:3, 3] = (np.array(mesh_xyz)).reshape(1,3)
    return mesh_transform

def ros_path2rosolute_path(ros_path,ros_path_prefix):#ros_path 是记录在urdf中的ros_path
    parts = ros_path.split('/')
    parts[-1] = parts[-1].replace("dae", "obj")
    parts = parts[2:]
    rosolute_path = ros_path_prefix / Path('/'.join(parts))
    return rosolute_path

def scale_inittransform_read_obj(attrib,mesh_origin,ros_path_prefix):
    #scale:np.array((1x3))
    file_path = ros_path2rosolute_path(attrib['filename'],ros_path_prefix)
    mesh = o3d.io.read_triangle_mesh(str(file_path))
    mesh.compute_vertex_normals()#recalculate the normal vector to reder in the open3d
    vertices = np.asarray(mesh.vertices).copy()
    if 'scale' in attrib.keys():
        scale = np.array([float(item) for item in attrib['scale'].split(' ')]).reshape(1,3).squeeze(0)
        vertices = vertices * scale
        
    if mesh_origin is not None:
        origin_rpy = mesh_origin.attrib['rpy'].split(' ')
        origin_xyz = mesh_origin.attrib['xyz'].split(' ')
        origin_rpy = [float(item) for item in origin_rpy]   
        origin_xyz = [float(item) for item in origin_xyz]   
        mesh_transform = euler_translation2transform(origin_rpy,origin_xyz)[:3,:]#不需要齐次的最后一行
        vertices = mesh_transform @ np.hstack((vertices,np.ones((vertices.shape[0],1)))).T
        vertices = vertices.T
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    return mesh

In [9]:
def get_mesh_from_urdf(urdf_path,ros_names:set,ros_path_prefix):
    urdf_tree = ET.parse(urdf_path)
    #names:mesh + position
    root = urdf_tree.getroot()
    links = root.findall('link')
    link_names = set(link.attrib['name'] for link in links)
    mesh_list = {}
    for link in links:#少加载了一个link，就是mouting
        name = link.attrib['name']
        visual = link.find('visual')
        if name in ros_names and visual is not None:
            geometry = visual.find('geometry')
            mesh_origin = visual.find('origin')
            if geometry is not None:
                mesh = geometry.find('mesh')
                if mesh is not None:
                    mesh_list[name] = scale_inittransform_read_obj(mesh.attrib,mesh_origin,ros_path_prefix)
        
        link_visuals = link.findall('visual')
        if len(link_visuals) > 1 :
            visual = link_visuals[1]
            if name in ros_names and visual is not None:
                geometry = visual.find('geometry')
                mesh_origin = visual.find('origin')
                if geometry is not None:
                    name = geometry.attrib['name']
                    mesh = geometry.find('mesh')
                    if mesh is not None:
                        mesh_list[name] = scale_inittransform_read_obj(mesh.attrib,mesh_origin,ros_path_prefix)
    #得到所有带有名字的mesh
    return  mesh_list,link_names#这个link_name没有用，因为它记录的是所有的urdf中的name,但是mesh_list中的是有用的

setattr(param_collect,"mesh_list",None)
setattr(param_collect,"link_names",None)
param_collect.mesh_list, param_collect.link_names = get_mesh_from_urdf(path_data.urdf_path,param_collect.global_name_postion.keys(),path_data.ros_path_prefix)

In [10]:
def seven_num2matrix(translation,roatation):#translation x,y,z rotation x,y,z,w
    transform_matrix = np.identity(4)
    transform_matrix[:3,:3] = Rotation.from_quat(roatation).as_matrix()
    transform_matrix[:3,3] = translation
    return transform_matrix


def transform_mesh_with_matrix(transform_matrix,mesh):
    vertices = np.asarray(mesh.vertices).copy() 
    vertices = np.hstack((vertices,np.ones((vertices.shape[0],1))),dtype=float).T
    transformed_vertices = np.dot(transform_matrix[:3,:],vertices).T
    mesh.vertices = o3d.utility.Vector3dVector(transformed_vertices)
    return mesh

In [11]:
# def seven_num2matrix(seven_num):#translation x,y,z rotation x,y,z,w
#     translation = seven_num[:3]
#     roatation = seven_num[3:]
#     transform_matrix = np.identity(4)
#     transform_matrix[:3,:3] = Rotation.from_quat(roatation).as_matrix()
#     transform_matrix[:3,3] = translation
#     return transform_matrix


# world2ra_base_link = np.array([0.0, 0.0, 0.7551, 0.0, 0.0, 0.0, 1.0])
# ra_base_link2ra_base_link_inertia = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 6.123233995736766e-17])
# ra_base_link_inertia2ra_shoulder_link = np.array([0.0, 0.0, 0.180799713106959, 0.0, 0.0, -0.02218457499886938, 0.9997538920314938])
# ra_shoulder_link2ra_upper_arm_link = np.array([2.971310608630752e-05, 0.0, 0.0, -0.48996396968907335, -0.5096328683978227, 0.5098448865795254, -0.4901712348596587])
# ra_upper_arm_link2ra_forearm_link = np.array([-0.6122699596353224, 0.0, 0.0, 0.0005534882508678976, 7.810096367055398e-06, 0.9097220400476349, 0.41521741707366755])
# ra_forearm_link2ra_wrist_1_link = np.array([-0.5712235523145621, -0.0008084706542084543, 0.17396290265016, 0.0020020415622941925, 0.0012167823515713007, -0.40612840285885266, 0.9138130178880192])
# ra_wrist_1_link2ra_wrist_2_link = np.array([-6.228072584678401e-05 ,-0.1197351589322399, 0.0001732019896383589, 0.4797352495698863, -0.5187781990928146, 0.5195297605087111, 0.48042907740050045])
# ra_wrist_2_link2ra_wrist_3_link = np.array([-5.615114057926111e-06, 0.115498055324382, 0.0001591452770727006, 0.022044549991752826, 0.706275503098048, 0.7072493547590925, -0.022074911825592784])

# debug_dict = {"world":{"ra_base_link":np.array([0.0, 0.0, 0.7551, 0.0, 0.0, 0.0, 1.0])},
#               "ra_base_link":{"ra_base_link_inertia":np.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 6.123233995736766e-17]),
#                               "cam0_camera_base":np.array([ 1.9443764948855990493, -0.12431574911658915816, 0.5813889603910600279, -0.31853917771567580308, 0.13549869630556143907, 0.9303635240824751351, 0.12081642527633146278])},
#               "ra_base_link_inertia":{"ra_shoulder_link":np.array([0.0, 0.0, 0.180799713106959, 0.0, 0.0, -0.02218457499886938, 0.9997538920314938])},
#               "ra_shoulder_link":{"ra_upper_arm_link":np.array([2.971310608630752e-05, 0.0, 0.0, -0.48996396968907335, -0.5096328683978227, 0.5098448865795254, -0.4901712348596587])},
#               "ra_upper_arm_link":{"ra_forearm_link":np.array([-0.6122699596353224, 0.0, 0.0, 0.0005534882508678976, 7.810096367055398e-06, 0.9097220400476349, 0.41521741707366755])},
#               "ra_forearm_link":{"ra_wrist_1_link":np.array([-0.5712235523145621, -0.0008084706542084543, 0.17396290265016, 0.0020020415622941925, 0.0012167823515713007, -0.40612840285885266, 0.9138130178880192])},
#               "ra_wrist_1_link":{"ra_wrist_2_link" :np.array([-6.228072584678401e-05 ,-0.1197351589322399, 0.0001732019896383589, 0.4797352495698863, -0.5187781990928146, 0.5195297605087111, 0.48042907740050045])},
#               "ra_wrist_2_link":{"ra_wrist_3_link" :np.array([-5.615114057926111e-06, 0.115498055324382, 0.0001591452770727006, 0.022044549991752826, 0.706275503098048, 0.7072493547590925, -0.022074911825592784])},              
#               "cam0_camera_base":{ "cam0_depth_camera_link":np.array([0.0, 0.0, 0.0017999999690800905, 0.525482745498759, -0.5254827454987588, 0.473146789255815, -0.4731467892558148]),
#                                   "cam1_camera_base":np.array([0.4818945275294305719, 0.4749957153245707997, 0.089710719093824276005, 0.22859941537846267812, -0.107766342249975236034, -0.51073788731596470036, 0.8217515033309714667]),
#                                   "cam2_camera_base":np.array([1.0349860481445547489, -0.09454256748291976764, 0.5636582532834875092, -0.62507661268989811454, 0.13925486542508763721, 0.7651120252517612519, 0.06701417416272763272]),
#                                   "cam3_camera_base":np.array([0.5335642080393309117, -0.4729291325134079016, 0.46836104653117943686, -0.5053888335753381478, 0.10347827009526477937, 0.60622676338080638825, 0.60527967575021257574])
#                                   },
#              "cam0_depth_camera_link":{"cam0_rgb_camera_link":np.array([0.03195635929956476, 0.0027212072088709655, -0.003681032970613338, 0.0436957276111748, 0.00017274907392389528, 0.0004898676408116431, 0.9990447518469873])},
#               "cam1_camera_base":{"cam1_depth_camera_link":np.array([0.0, 0.0, 0.0017999999690800905228, 0.5254827454987589519, -0.5254827454987588409, 0.47314678925581499236, -0.47314678925581482583])},
#               "cam1_depth_camera_link":{"cam1_rgb_camera_link":np.array([0.03196861840242224556, 0.0025654099456104205272, -0.0037230595721786542002, 0.047327849588390161206, 0.00011795156847148814043, 0.00082686364986344403766, 0.99887906520113189934])}
#               }


# global_positon = defaultdict(dict)
# # #左乘
# # # global_positon["ra_base_link"] = seven_num2matrix(debug_dict["world"]["ra_base_link"])
# # # global_positon["ra_base_link_inertia" ] =seven_num2matrix(debug_dict["ra_base_link"]["ra_base_link_inertia"]) @global_positon["ra_base_link"]
# # # global_positon["ra_shoulder_link"] =seven_num2matrix(debug_dict["ra_base_link_inertia"]["ra_shoulder_link"])@global_positon["ra_base_link_inertia" ]
# # # global_positon["ra_upper_arm_link"] =seven_num2matrix(debug_dict["ra_shoulder_link"]["ra_upper_arm_link"])@global_positon["ra_shoulder_link"]
# # # global_positon["ra_forearm_link"] =seven_num2matrix(debug_dict["ra_upper_arm_link"]["ra_forearm_link"])@global_positon["ra_upper_arm_link"]
# # # global_positon["ra_wrist_1_link"] =seven_num2matrix(debug_dict["ra_forearm_link"]["ra_wrist_1_link"])@global_positon["ra_forearm_link"]
# # # global_positon["ra_wrist_2_link"] = seven_num2matrix(debug_dict["ra_wrist_1_link"]["ra_wrist_2_link"])@global_positon["ra_forearm_link"]
# # # global_positon["ra_wrist_3_link" ] =seven_num2matrix(debug_dict["ra_wrist_2_link"]["ra_wrist_3_link"])@global_positon["ra_wrist_2_link"]                        
# # #右乘
# global_positon["ra_base_link"] = seven_num2matrix(debug_dict["world"]["ra_base_link"])
# debug_dict["ra_base_link"]["ra_base_link_inertia"]
# global_positon["ra_base_link_inertia" ] =global_positon["ra_base_link"] @seven_num2matrix(debug_dict["ra_base_link"]["ra_base_link_inertia"]) 
# global_positon["ra_shoulder_link"] =global_positon["ra_base_link_inertia" ] @seven_num2matrix(debug_dict["ra_base_link_inertia"]["ra_shoulder_link"])
# global_positon["ra_upper_arm_link"] =global_positon["ra_shoulder_link"] @seven_num2matrix(debug_dict["ra_shoulder_link"]["ra_upper_arm_link"])
# global_positon["ra_forearm_link"] =global_positon["ra_upper_arm_link"]  @seven_num2matrix(debug_dict["ra_upper_arm_link"]["ra_forearm_link"])
# global_positon["ra_wrist_1_link"] =global_positon["ra_forearm_link"] @ seven_num2matrix(debug_dict["ra_forearm_link"]["ra_wrist_1_link"])
# global_positon["ra_wrist_2_link"] = global_positon["ra_wrist_1_link"] @ seven_num2matrix(debug_dict["ra_wrist_1_link"]["ra_wrist_2_link"])
# global_positon["ra_wrist_3_link" ] =global_positon["ra_wrist_2_link"]  @ seven_num2matrix(debug_dict["ra_wrist_2_link"]["ra_wrist_3_link"])
# global_positon["cam0_camera_base"] = global_positon["ra_base_link"] @ seven_num2matrix(debug_dict["ra_base_link"]["cam0_camera_base"])
# global_positon["cam0_depth_camera_link"] = global_positon["cam0_camera_base"] @ seven_num2matrix(debug_dict["cam0_camera_base"]["cam0_depth_camera_link"])
# global_positon["cam0_rgb_camera_link"] = global_positon["cam0_depth_camera_link"] @ seven_num2matrix(debug_dict["cam0_depth_camera_link"]["cam0_rgb_camera_link"])

# global_positon["cam1_camera_base"] = global_positon["cam0_camera_base"] @ seven_num2matrix(debug_dict["cam0_camera_base"]["cam1_camera_base"])
# global_positon["cam2_camera_base"] = global_positon["cam0_camera_base"] @ seven_num2matrix(debug_dict["cam0_camera_base"]["cam2_camera_base"])
# global_positon["cam3_camera_base"] = global_positon["cam0_camera_base"] @ seven_num2matrix(debug_dict["cam0_camera_base"]["cam3_camera_base"])
# global_positon["cam1_depth_camera_link"] = global_positon["cam1_camera_base"] @ seven_num2matrix(debug_dict["cam1_camera_base"]["cam1_depth_camera_link"])
# global_positon["cam1_rgb_camera_link"] = global_positon["cam1_depth_camera_link"] @ seven_num2matrix(debug_dict["cam1_depth_camera_link"]["cam1_rgb_camera_link"])
# # print(global_positon["ra_shoulder_link"])
# # print()
# # print(seven_num2matrix(debug_dict["ra_shoulder_link"]["ra_upper_arm_link"]))
# # print()
# # print(global_positon["ra_upper_arm_link"])
# print(global_positon["cam0_depth_camera_link"])



# ### 
# # [[ 0.99503791  0.09431951 -0.03167642  1.37615923]
# #  [ 0.07985236 -0.5671025   0.81976725 -0.47088446]
# #  [ 0.05935627 -0.81822893 -0.57182012  1.18806754]
# #  [ 0.          0.          0.          1.        ]]

In [12]:
# def vis_addline(vis,name):
#     points = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]])
#     points2 = ((global_positon[name] @ np.hstack((points,np.ones((4,1)))).T)[:3,:]).T
#     lines2 = o3d.geometry.LineSet()
#     lines2.points = o3d.utility.Vector3dVector(points2)
#     lines2.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
#     lines2.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
#     vis.add_geometry(lines2)


In [13]:
# links_names = [key for key in global_positon]

# merged_mesh = o3d.geometry.TriangleMesh()
# # for value in param_collect.mesh_list.values():
# #     merged_mesh += value
# for key,value in param_collect.mesh_list.items():
#     if key in links_names:
#         merged_mesh +=  transform_mesh_with_matrix(global_positon[key],param_collect.mesh_list[key])

# vis = o3d.visualization.Visualizer()
# vis.create_window()

# # 将mesh添加到可视化窗口中
# vis.add_geometry(merged_mesh)

# points = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]])
# lines = o3d.geometry.LineSet()
# lines.points = o3d.utility.Vector3dVector(points)
# lines.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
# lines.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
# vis.add_geometry(lines)

# points1 = ((global_positon["ra_base_link"] @ np.hstack((points,np.ones((4,1)))).T)[:3,:]).T
# lines1 = o3d.geometry.LineSet()
# lines1.points = o3d.utility.Vector3dVector(points1)
# lines1.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
# lines1.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
# vis.add_geometry(lines1)


# points2 = ((global_positon["ra_base_link_inertia" ] @ np.hstack((points,np.ones((4,1)))).T)[:3,:]).T
# lines2 = o3d.geometry.LineSet()
# lines2.points = o3d.utility.Vector3dVector(points2)
# lines2.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
# lines2.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
# vis.add_geometry(lines2)


# points3 = ((global_positon["ra_shoulder_link"] @ np.hstack((points,np.ones((4,1)))).T)[:3,:]).T
# lines3 = o3d.geometry.LineSet()
# lines3.points = o3d.utility.Vector3dVector(points3)
# lines3.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
# lines3.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
# vis.add_geometry(lines3)


# points4 = ((global_positon["ra_upper_arm_link"] @ np.hstack((points,np.ones((4,1)))).T)[:3,:]).T
# lines4 = o3d.geometry.LineSet()
# lines4.points = o3d.utility.Vector3dVector(points4)
# lines4.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
# lines4.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
# vis.add_geometry(lines4)


# points5 = ((global_positon["cam0_camera_base"] @ np.hstack((points,np.ones((4,1)))).T)[:3,:]).T
# lines5 = o3d.geometry.LineSet()
# lines5.points = o3d.utility.Vector3dVector(points5)
# lines5.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
# lines5.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
# vis.add_geometry(lines5)





# points6 = ((global_positon["cam0_depth_camera_link"] @ np.hstack((points,np.ones((4,1)))).T)[:3,:]).T
# lines6 = o3d.geometry.LineSet()
# lines6.points = o3d.utility.Vector3dVector(points6)
# lines6.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
# lines6.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
# vis.add_geometry(lines6)

# points7 = ((global_positon["cam0_rgb_camera_link"] @ np.hstack((points,np.ones((4,1)))).T)[:3,:]).T
# lines7 = o3d.geometry.LineSet()
# lines7.points = o3d.utility.Vector3dVector(points7)
# lines7.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
# lines7.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
# vis.add_geometry(lines7)


# points8 = ((global_positon["cam0_camera_base"] @ np.hstack((points,np.ones((4,1)))).T)[:3,:]).T
# lines8 = o3d.geometry.LineSet()
# lines8.points = o3d.utility.Vector3dVector(points8)
# lines8.lines = o3d.utility.Vector2iVector(np.array([[0, 1], [0, 2], [0, 3]]))
# lines8.colors = o3d.utility.Vector3dVector(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))
# vis.add_geometry(lines)

# vis_addline(vis,"cam1_camera_base")
# vis_addline(vis,"cam2_camera_base")
# vis_addline(vis,"cam3_camera_base")


# vis.run()
# vis.destroy_window()




In [14]:
def find_time_closet(slot,time_stamps):
    diff = np.abs(time_stamps - slot)
    index = np.argmin(diff)
    return index


def dfs_position(TF_tree,global_name_postion,time_slot):
    global_name_postion['world'] = np.identity(4)
    dfs_position_update(TF_tree,global_name_postion,'world',time_slot)
    global_name_postion["rh_mounting_plate"] = global_name_postion["rh_forearm"]#这里面有一个安装盘，这个安装盘的位置是跟机械化艘的手腕的位置是一样的，特殊设置
    
    
def transform_mesh_with_quater(senven_num_transform,mesh):
    child_transform = seven_num2matrix(senven_num_transform[:3],senven_num_transform[3:])[:3,:]
    vertices = np.asarray(mesh.vertices).copy() 
    vertices = np.hstack([vertices,np.ones((vertices.shape[0],1))],dtype=float).T
    transformed_vertices = np.dot(child_transform,vertices).T
    mesh.vertices = o3d.utility.Vector3dVector(transformed_vertices)
    return mesh


def transform_mesh_with_matrix(transform_matrix,mesh):
    vertices = np.asarray(mesh.vertices).copy() 
    vertices = np.hstack((vertices,np.ones((vertices.shape[0],1))),dtype=float).T
    transformed_vertices = np.dot(transform_matrix[:3,:],vertices).T
    mesh.vertices = o3d.utility.Vector3dVector(transformed_vertices)
    return mesh


def dfs_position_update(TF_tree,global_name_postion,name,time_slot):
    if name in TF_tree.keys():#叶子节点不会在keys里面
        for child_name in TF_tree[name].keys():
            child_time_and_transform = TF_tree[name][child_name]#有些TF只有一个，是static TF
            time_index = find_time_closet(time_slot,child_time_and_transform[:,0])
            senven_num_transform = child_time_and_transform[time_index,1:]
            child_transform = seven_num2matrix(translation=senven_num_transform[:3],roatation=senven_num_transform[3:])
            child_transform_position = np.dot(global_name_postion[name],child_transform)
            global_name_postion[child_name] = child_transform_position
            dfs_position_update(TF_tree,global_name_postion,child_name,time_slot)


def show_meshes(meshes):
    o3d.visualization.draw_geometries(meshes)

def output_merge_mesh(meshes,output_path):
    merged_mesh = o3d.geometry.TriangleMesh()
    for key,value in meshes.items():
        # if key.startswith("rh"):
        merged_mesh += value
    o3d.io.write_triangle_mesh(str(output_path), merged_mesh)

def save_global_position(global_name_postion,num,gobal_position_folder):# attention some item's transform is None
    if not os.path.exists(gobal_position_folder):
        os.makedirs(gobal_position_folder)
    with open(gobal_position_folder / Path(f"{num}.txt"), "w+") as position_write:
        global_name_postion = {key: value.tolist() if isinstance(value, np.ndarray) else value for key, value in global_name_postion.items()}
        json.dump(global_name_postion, position_write, indent=4)

def show_meshposition(rgb_timestamp,mesh_list,TF_tree,global_name_postion,output_folder,base_frame,pinhole_camera_parameters_path,gobal_position_folder,output_jud,positon_vis = False):#最后的参数表示，我们这个展示的点云是要在哪个坐标系下表示
    vis = None
    if positon_vis:
        vis = o3d.visualization.Visualizer()
        vis.create_window()
    camera_params = o3d.io.read_pinhole_camera_parameters(str(pinhole_camera_parameters_path))
    for num,slot in enumerate(rgb_timestamp):
        print(num)
        dfs_position(TF_tree,global_name_postion,slot)
        save_global_position(global_name_postion,num,gobal_position_folder)
        mesh_show = copy.deepcopy(mesh_list)
        for mesh_name in mesh_show.keys():
            mesh_show[mesh_name] = transform_mesh_with_matrix(global_name_postion[mesh_name],mesh_show[mesh_name])
        if base_frame != 'world':
            for mesh_name in mesh_show.keys():
                mesh_show[mesh_name] = transform_mesh_with_matrix(global_name_postion[base_frame],mesh_show[mesh_name])
        # the code for show point cloud sequence
        if positon_vis:
            if num == 0:
                for mesh_item in mesh_show.values():
                    vis.add_geometry(mesh_item)
            else:
                vis.clear_geometries()
                for mesh_item in mesh_show.values():
                    vis.add_geometry(mesh_item)
                vis.get_view_control().convert_from_pinhole_camera_parameters(camera_params)
                vis.poll_events()
                vis.update_renderer()
            vis.run()  # 这将显示mesh并允许交互直到用户按'q'

        if output_jud:
            if not os.path.exists(output_folder):
                os.makedirs(output_folder)
            output_merge_mesh(mesh_show,output_folder / Path(base_frame+"_"+ str(num) + ".obj"))
    # vis.destroy_window()
        # show_meshes([value for value in mesh_show.values()])


show_meshposition(rgb_timestamp = param_collect.rgb_timestamp,
                  mesh_list = param_collect.mesh_list,
                  TF_tree = param_collect.TF_tree,
                  global_name_postion = param_collect.global_name_postion,
                  output_folder = path_data.merge_mesh_output_folder,
                  base_frame = 'world',
                  pinhole_camera_parameters_path = path_data.pinhole_camera_parameters_path,
                  gobal_position_folder = path_data.gobal_position_folder,
                  output_jud = True)

#修改观看视角将导致之后的机械手有可能动不了

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
